# CNW-Grass gas exchange for developers, 2: extend the component

**What you will learn:** add a state variable and a step to `GasExchange` by subclassing it, run it in a plant model
and a scene, and test it.

**You need:** developer tutorial 1 (the component formalism). **Run time:** a few seconds.

## 1. A new variable and its step

Each element's **water use efficiency**, the CO₂ it fixes per water it loses: `An / Tr` (µmol of CO₂ per mmol of
water), 0 where it transpires nothing. A subclass declares it and computes it in a step:

- `state_variable` declares it: its unit, its description, how it is aggregated (`intensive`, weighted by the green
  area over coarser scales, as `An`), its initial value and its scale (`SubOrgan`);
- the method `_water_use_efficiency` computes it. Its arguments name the variables it reads, the dictionary it
  returns the variables it writes;
- `@totalstate(include_inactive=True)` places it in the row "totalstate", after `_exchange` (row "state");
- `@dataclass` is needed on the subclass, as on every component.

In [1]:
from dataclasses import dataclass

import numpy as np

from openalea.cnwgrass.metafspm.gas_exchange import GasExchange
from openalea.metafspm.coupling.component import state_variable
from openalea.metafspm.data_structure.configs import ScalesConfig
from openalea.metafspm.solve.decorator import totalstate


@dataclass
class GasExchangeWithEfficiency(GasExchange):
    water_use_efficiency: float = state_variable(unit="µmol mmol-1", unit_comment="of CO2 per water",
                                                 description="The element's net assimilation per water transpired",
                                                 kind="intensive", weight="green_area", default=0.,
                                                 scale=ScalesConfig.SubOrgan)

    @totalstate(include_inactive=True)
    def _water_use_efficiency(self, An, Tr) -> dict:
        An, Tr = np.asarray(An, dtype=float), np.asarray(Tr, dtype=float)
        return {"water_use_efficiency": np.divide(An, Tr, out=np.zeros_like(An), where=Tr > 0)}

## 2. Where it appears

The new variable is in the declarations, the new step in the schedule:

In [2]:
declarations = GasExchangeWithEfficiency.declarations()
print(declarations[declarations["name"] == "water_use_efficiency"][["name", "kind", "scale", "unit"]].to_string())
print()
for stages, steps in GasExchangeWithEfficiency.schedule():
    print(f"{', '.join(stages):>10}  {', '.join(steps)}")

                    name       kind     scale         unit
98  water_use_efficiency  intensive  SubOrgan  µmol mmol-1

      rate  _canopy_height, _capacities
     state  _exchange
totalstate  _water_use_efficiency


## 3. In a plant model and a scene

The plant model is the users' `ExchangingWheat` with the extended component in place of `GasExchange` (its class
attribute `gas_exchange_component`). The links name the components by their class: `adel_links` takes the
receiver's name. The scene holds the plants and Caribu's light.

In [3]:
from openalea.adel.development import random_delays, sown_stand
from openalea.adel.leaf_shapes import LeafShapes
from openalea.caribu.component import CaribuLightModel
from openalea.cnwgrass.metafspm.gas_exchange import adel_links
from openalea.cnwgrass.metafspm.simulation import ExchangingWheat
from openalea.metafspm.scene.population import planting
from openalea.metafspm.scene.scene import Scene


class EfficientWheat(ExchangingWheat):
    gas_exchange_component = GasExchangeWithEfficiency
    translator = adel_links(receiver="GasExchangeWithEfficiency")


shoot = {"n_sect": 2, "leaves": LeafShapes.echap(xy_model="Soissons_byleafclass"), "start": 600.}
table = planting(sown_stand(8, n_rows=2), EfficientWheat, shoot,
                 [{"delay": float(delay)} for delay in random_delays(8)], xrange=0.16, yrange=0.3)
weather = {"PARi": 1500., "air_temperature": 20., "relative_humidity": 0.6, "ambient_CO2": 410., "wind_speed": 2.}
scene = Scene(table, environment=[CaribuLightModel], environment_scenarios=[{"sun": True}], time_step=3600.,
              forcings=weather, start=(130, 11.))
scene.populations[0].instance.grow()
scene.run()
ds = scene.populations[0].data_structure
efficiency = np.asarray(ds.get("water_use_efficiency"))
green = np.asarray(ds.get("green_area"))
print(f"{np.sum(efficiency > 0)} elements fix CO2; their water use efficiency, from {efficiency[efficiency > 0].min():.2f} "
      f"to {efficiency.max():.2f} µmol mmol-1; the plants' (green-area weighted) "
      f"{np.round(ds.map_to(efficiency, 'node', 'Plant', 'weighted_mean', weights=green), 2)}")

380 elements fix CO2; their water use efficiency, from 0.03 to 7.20 µmol mmol-1; the plants' (green-area weighted) [5.67 5.52 5.54 5.63 5.46 5.73 5.64 5.39]


## 4. Test it

A test states what the step must give, on an independent computation. In the package it would go in `test/`, run by
pytest; here it runs in place:

In [4]:
def test_the_efficiency_is_An_over_Tr():
    An, Tr = np.asarray(ds.get("An")), np.asarray(ds.get("Tr"))
    transpiring = Tr > 0
    np.testing.assert_allclose(efficiency[transpiring], An[transpiring] / Tr[transpiring], rtol=1e-12)
    assert (efficiency[~transpiring] == 0).all()
    assert transpiring.sum() > 20


test_the_efficiency_is_An_over_Tr()
print("passed")

passed


## 5. Further

- A variable another model writes is declared with `input_variable(..., by=...)`: `GasExchange`'s `surfacic_nitrogen`
  and `surfacic_NSC` are inputs a CN model will give (`surfacic_photosynthetic_nitrogen`, `surfacic_total_nitrogen`,
  `surfacic_nonstructural_carbohydrates` compute them from its amounts, as the legacy).
- A new stomatal model is a change of the equations, not a new step: `stomatal_conductance` (in
  `openalea.cnwgrass.metafspm.gas_exchange`) and the setting's `choices` (`STOMATAL_MODELS`), with its equation in the
  module's docstring and a test against an independent computation.
- An output in another unit or under another name for another model is a translator link (developer tutorial 1,
  section 5), not a new variable.